In [0]:
spark.sql("""
          CREATE SCHEMA IF NOT EXISTS
          dbw_netflix_de_2026.netflix_bronze
          """)

print("Bronze schema ready")

Bronze schema ready


In [0]:

# Source: Netflix CSV in ADLS Raw
raw_path = "abfss://raw@stnetflix.dfs.core.windows.net/"

# Auto Loader schema tracking
schema_path = (
    "abfss://bronze@stnetflix.dfs.core.windows.net/"
    "_schemas/netflix_titles/"
)

# Streaming checkpoint
checkpoint_path = (
    "abfss://bronze@stnetflix.dfs.core.windows.net/"
    "_checkpoints/netflix_titles/"
)

# Destination Bronze Delta table
bronze_table = (
    "dbw_netflix_de_2026.netflix_bronze.netflix_titles"
)

print("Auto Loader paths configured!")


Auto Loader paths configured!


In [0]:

df_autoloader = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.useManagedFileEvents", "false")
    .option("header", "true")
    .option("multiLine", "true")
    .option("cloudFiles.inferColumnTypes", "false")
    .load(raw_path)
)

print("Auto Loader configured successfully!")


Auto Loader configured successfully!


In [0]:

query = (
    df_autoloader.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()

print("Bronze ingestion completed!")


Bronze ingestion completed!


In [0]:

df_bronze = spark.table(bronze_table)

print("Bronze record count:", df_bronze.count())

display(df_bronze.limit(10))


Bronze record count: 6236


duration_minutes,duration_seasons,type,title,date_added,release_year,rating,description,show_id,_rescued_data
90,null,Movie,Norm of the North: King Sized Adventure,9/9/2019,2019,TV-PG,"Before planning an awesome wedding for his grandfather, a polar bear king must take back a stolen artifact from an evil archaeologist first.",81145628,null
94,null,Movie,Jandino: Whatever it Takes,9/9/2016,2016,TV-MA,"""Jandino Asporaat riffs on the challenges of raising kids and serenades the audience with a rousing rendition of """"Sex on Fire"""" in his comedy show.""",80117401,null
null,1,TV Show,Transformers Prime,9/8/2018,2013,TV-Y7-FV,"With the help of three human allies, the Autobots once again protect Earth from the onslaught of the Decepticons and their leader, Megatron.",70234439,null
null,1,TV Show,Transformers: Robots in Disguise,9/8/2018,2016,TV-Y7,"When a prison ship crash unleashes hundreds of Decepticons on Earth, Bumblebee leads a new Autobot force to protect humankind.",80058654,null
99,null,Movie,#realityhigh,9/8/2017,2017,TV-14,"When nerdy high schooler Dani finally attracts the interest of her longtime crush, she lands in the cross hairs of his ex, a social media celebrity.",80125979,null
null,1,TV Show,Apaches,9/8/2017,2016,TV-MA,A young journalist is forced into a life of crime to save his father and family in this series based on the novel by Miguel Sáez Carral.,80163890,null
110,null,Movie,Automata,9/8/2017,2014,R,"In a dystopian future, an insurance adjuster for a tech company investigates a robot killed for violating protocol and discovers a global conspiracy.",70304989,null
60,null,Movie,Fabrizio Copano: Solo pienso en mi,9/8/2017,2017,TV-MA,"Fabrizio Copano takes audience participation to the next level in this stand-up set while reflecting on sperm banks, family WhatsApp groups and more.",80164077,null
null,1,TV Show,Fire Chasers,9/8/2017,2017,TV-MA,"As California's 2016 fire season rages, brave backcountry firefighters race to put out the flames, protect homes and save lives in this docuseries.",80117902,null
90,null,Movie,Good People,9/8/2017,2014,R,A struggling couple can't believe their luck when they find a stash of money in the apartment of a neighbor who was recently murdered.,70304990,null


In [0]:

bronze_table = "dbw_netflix_de_2026.netflix_bronze.netflix_titles"

df_bronze = spark.table(bronze_table)

print("Total Bronze Records:", df_bronze.count())
print("Total Columns:", len(df_bronze.columns))

df_bronze.printSchema()
display(df_bronze.limit(10))


Total Bronze Records: 6236
Total Columns: 10
root
 |-- duration_minutes: string (nullable = true)
 |-- duration_seasons: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- release_year: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- description: string (nullable = true)
 |-- show_id: string (nullable = true)
 |-- _rescued_data: string (nullable = true)



duration_minutes,duration_seasons,type,title,date_added,release_year,rating,description,show_id,_rescued_data
90,null,Movie,Norm of the North: King Sized Adventure,9/9/2019,2019,TV-PG,"Before planning an awesome wedding for his grandfather, a polar bear king must take back a stolen artifact from an evil archaeologist first.",81145628,null
94,null,Movie,Jandino: Whatever it Takes,9/9/2016,2016,TV-MA,"""Jandino Asporaat riffs on the challenges of raising kids and serenades the audience with a rousing rendition of """"Sex on Fire"""" in his comedy show.""",80117401,null
null,1,TV Show,Transformers Prime,9/8/2018,2013,TV-Y7-FV,"With the help of three human allies, the Autobots once again protect Earth from the onslaught of the Decepticons and their leader, Megatron.",70234439,null
null,1,TV Show,Transformers: Robots in Disguise,9/8/2018,2016,TV-Y7,"When a prison ship crash unleashes hundreds of Decepticons on Earth, Bumblebee leads a new Autobot force to protect humankind.",80058654,null
99,null,Movie,#realityhigh,9/8/2017,2017,TV-14,"When nerdy high schooler Dani finally attracts the interest of her longtime crush, she lands in the cross hairs of his ex, a social media celebrity.",80125979,null
null,1,TV Show,Apaches,9/8/2017,2016,TV-MA,A young journalist is forced into a life of crime to save his father and family in this series based on the novel by Miguel Sáez Carral.,80163890,null
110,null,Movie,Automata,9/8/2017,2014,R,"In a dystopian future, an insurance adjuster for a tech company investigates a robot killed for violating protocol and discovers a global conspiracy.",70304989,null
60,null,Movie,Fabrizio Copano: Solo pienso en mi,9/8/2017,2017,TV-MA,"Fabrizio Copano takes audience participation to the next level in this stand-up set while reflecting on sperm banks, family WhatsApp groups and more.",80164077,null
null,1,TV Show,Fire Chasers,9/8/2017,2017,TV-MA,"As California's 2016 fire season rages, brave backcountry firefighters race to put out the flames, protect homes and save lives in this docuseries.",80117902,null
90,null,Movie,Good People,9/8/2017,2014,R,A struggling couple can't believe their luck when they find a stash of money in the apartment of a neighbor who was recently murdered.,70304990,null


In [0]:

from pyspark.sql.functions import col

print("Total Records:", df_bronze.count())

print(
    "Distinct Show IDs:",
    df_bronze.select("show_id").distinct().count()
)

print(
    "Null Show IDs:",
    df_bronze.filter(col("show_id").isNull()).count()
)

print(
    "Rescued Records:",
    df_bronze.filter(
        col("_rescued_data").isNotNull()
    ).count()
)


Total Records: 6236
Distinct Show IDs: 6233
Null Show IDs: 4
Rescued Records: 0


In [0]:

from pyspark.sql.functions import col, count

duplicates = (
    df_bronze
    .filter(col("show_id").isNotNull())
    .groupBy("show_id")
    .agg(count("*").alias("record_count"))
    .filter(col("record_count") > 1)
)

display(duplicates)


show_id,record_count


In [0]:
spark.sql("""
          CREATE SCHEMA IF NOT EXISTS
          dbw_netflix_de_2026.netflix_silver
          """)

print("silver schema is ready")

silver schema is ready


In [0]:

# Load Bronze Delta table into Silver notebook
bronze_table = "dbw_netflix_de_2026.netflix_bronze.netflix_titles"

df_bronze = spark.table(bronze_table)

print("Bronze data loaded successfully!")
print("Total Records:", df_bronze.count())


Bronze data loaded successfully!
Total Records: 6236


In [0]:


from pyspark.sql.functions import (
    col, trim, current_timestamp, expr
)

bronze_table = "dbw_netflix_de_2026.netflix_bronze.netflix_titles"
df_bronze = spark.table(bronze_table)

df_silver = (
    df_bronze
    .filter(
        col("show_id").isNotNull() &
        (trim(col("show_id")) != "")
    )
    .dropDuplicates(["show_id"])
    .withColumn("title", trim(col("title")))
    .withColumn("type", trim(col("type")))
    .withColumn(
        "date_added",
        expr("try_to_timestamp(trim(date_added), 'M/d/yyyy')").cast("date")
    )
    .withColumn(
        "release_year",
        expr("try_cast(release_year AS INT)")
    )
    .withColumn(
        "duration_minutes",
        expr("try_cast(duration_minutes AS INT)")
    )
    .withColumn(
        "duration_seasons",
        expr("try_cast(duration_seasons AS INT)")
    )
    .withColumn("processed_at", current_timestamp())
)

display(df_silver.limit(10))


duration_minutes,duration_seasons,type,title,date_added,release_year,rating,description,show_id,_rescued_data,processed_at
90,null,Movie,Norm of the North: King Sized Adventure,2019-09-09,2019,TV-PG,"Before planning an awesome wedding for his grandfather, a polar bear king must take back a stolen artifact from an evil archaeologist first.",81145628,null,2026-11-24T03:29:28.040Z
94,null,Movie,Jandino: Whatever it Takes,2016-09-09,2016,TV-MA,"""Jandino Asporaat riffs on the challenges of raising kids and serenades the audience with a rousing rendition of """"Sex on Fire"""" in his comedy show.""",80117401,null,2026-11-24T03:29:28.040Z
null,1,TV Show,Transformers Prime,2018-09-08,2013,TV-Y7-FV,"With the help of three human allies, the Autobots once again protect Earth from the onslaught of the Decepticons and their leader, Megatron.",70234439,null,2026-11-24T03:29:28.040Z
null,1,TV Show,Transformers: Robots in Disguise,2018-09-08,2016,TV-Y7,"When a prison ship crash unleashes hundreds of Decepticons on Earth, Bumblebee leads a new Autobot force to protect humankind.",80058654,null,2026-11-24T03:29:28.040Z
99,null,Movie,#realityhigh,2017-09-08,2017,TV-14,"When nerdy high schooler Dani finally attracts the interest of her longtime crush, she lands in the cross hairs of his ex, a social media celebrity.",80125979,null,2026-11-24T03:29:28.040Z
null,1,TV Show,Apaches,2017-09-08,2016,TV-MA,A young journalist is forced into a life of crime to save his father and family in this series based on the novel by Miguel Sáez Carral.,80163890,null,2026-11-24T03:29:28.040Z
110,null,Movie,Automata,2017-09-08,2014,R,"In a dystopian future, an insurance adjuster for a tech company investigates a robot killed for violating protocol and discovers a global conspiracy.",70304989,null,2026-11-24T03:29:28.040Z
60,null,Movie,Fabrizio Copano: Solo pienso en mi,2017-09-08,2017,TV-MA,"Fabrizio Copano takes audience participation to the next level in this stand-up set while reflecting on sperm banks, family WhatsApp groups and more.",80164077,null,2026-11-24T03:29:28.040Z
null,1,TV Show,Fire Chasers,2017-09-08,2017,TV-MA,"As California's 2016 fire season rages, brave backcountry firefighters race to put out the flames, protect homes and save lives in this docuseries.",80117902,null,2026-11-24T03:29:28.040Z
90,null,Movie,Good People,2017-09-08,2014,R,A struggling couple can't believe their luck when they find a stash of money in the apartment of a neighbor who was recently murdered.,70304990,null,2026-11-24T03:29:28.040Z


In [0]:

print("Bronze records:", df_bronze.count())
print("Silver records:", df_silver.count())

df_silver.select(
    "show_id",
    "title",
    "date_added",
    "release_year",
    "duration_minutes",
    "duration_seasons"
).show(10, truncate=False)


Bronze records: 6236
Silver records: 6232
+--------+---------------------------------------+----------+------------+----------------+----------------+
|show_id |title                                  |date_added|release_year|duration_minutes|duration_seasons|
+--------+---------------------------------------+----------+------------+----------------+----------------+
|81145628|Norm of the North: King Sized Adventure|NULL      |2019        |90              |NULL            |
|80117401|Jandino: Whatever it Takes             |NULL      |2016        |94              |NULL            |
|70234439|Transformers Prime                     |NULL      |2013        |NULL            |1               |
|80058654|Transformers: Robots in Disguise       |NULL      |2016        |NULL            |1               |
|80125979|#realityhigh                           |NULL      |2017        |99              |NULL            |
|80163890|Apaches                                |NULL      |2016        |NULL        

In [0]:
spark.sql("""
          CREATE SCHEMA IF NOT EXISTS
          dbw_netflix_de_2026.netflix_silver
          """)

silver_table = "dbw_netflix_de_2026.netflix_silver.netflix_titles"

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

print("silver schema is ready")
print("silver records:", spark.table(silver_table).count())

silver schema is ready
silver records: 6232


In [0]:

from pyspark.sql.functions import col

df_silver_check = spark.table(
    "dbw_netflix_de_2026.netflix_silver.netflix_titles"
)

print(
    "Valid dates:",
    df_silver_check.filter(col("date_added").isNotNull()).count()
)

print(
    "Null dates:",
    df_silver_check.filter(col("date_added").isNull()).count()
)

display(
    df_silver_check.select("show_id", "title", "date_added").limit(10)
)


Valid dates: 0
Null dates: 6232


show_id,title,date_added
80182482,Krish Trish and Baltiboy: Comics of India,null
81016750,The 3rd Eye,null
81186758,Oh! Baby (Malayalam),null
80162144,White Island,null
80190449,The Endless,null
81030342,Los Tigres del Norte at Folsom Prison,null
80241102,Cabins in the Wild with Dick Strawbridge,null
80240816,Inside The Freemasons,null
80142637,Janaan,null
80243899,Behind the Newsroom,null


In [0]:

from pyspark.sql.functions import col

bronze = spark.table(
    "dbw_netflix_de_2026.netflix_bronze.netflix_titles"
)

silver = spark.table(
    "dbw_netflix_de_2026.netflix_silver.netflix_titles"
)

print(
    "Bronze valid dates:",
    bronze.filter(col("date_added").isNotNull()).count()
)

print(
    "Silver valid dates:",
    silver.filter(col("date_added").isNotNull()).count()
)

bronze.select("date_added").show(10, truncate=False)


Bronze valid dates: 6223
Silver valid dates: 0
+----------+
|date_added|
+----------+
|9/9/2019  |
|9/9/2016  |
|9/8/2018  |
|9/8/2018  |
|9/8/2017  |
|9/8/2017  |
|9/8/2017  |
|9/8/2017  |
|9/8/2017  |
|9/8/2017  |
+----------+
only showing top 10 rows


In [0]:

from pyspark.sql.functions import col

silver = spark.table(
    "dbw_netflix_de_2026.netflix_silver.netflix_titles"
)

print("Silver records:", silver.count())

print(
    "Valid Silver dates:",
    silver.filter(col("date_added").isNotNull()).count()
)

display(
    silver.select(
        "show_id", "title", "date_added"
    ).limit(10)
)


Silver records: 6232
Valid Silver dates: 0


show_id,title,date_added
80182482,Krish Trish and Baltiboy: Comics of India,null
81016750,The 3rd Eye,null
81186758,Oh! Baby (Malayalam),null
80162144,White Island,null
80190449,The Endless,null
81030342,Los Tigres del Norte at Folsom Prison,null
80241102,Cabins in the Wild with Dick Strawbridge,null
80240816,Inside The Freemasons,null
80142637,Janaan,null
80243899,Behind the Newsroom,null


In [0]:

from pyspark.sql.functions import col, expr, trim

bronze = spark.table(
    "dbw_netflix_de_2026.netflix_bronze.netflix_titles"
)

df_date_test = bronze.select(
    "show_id",
    "title",
    col("date_added").alias("original_date"),
    expr(
        "try_to_timestamp(trim(date_added), 'M/d/yyyy')"
    ).cast("date").alias("converted_date")
)

print("Total Bronze records:", df_date_test.count())

print(
    "Successfully converted dates:",
    df_date_test.filter(
        col("converted_date").isNotNull()
    ).count()
)

display(
    df_date_test.filter(
        col("original_date").isNotNull() &
        col("converted_date").isNull()
    ).limit(20)
)


Total Bronze records: 6236
Successfully converted dates: 6222


show_id,title,original_date,converted_date
null,TV-PG,This documentary centers on the crew of the B-17 Flying Fortress Memphis Belle as it prepares to execute a strategic bombing mission over Germany.,null


In [0]:

from pyspark.sql.functions import col

silver_table = "dbw_netflix_de_2026.netflix_silver.netflix_titles"

print("Silver records:", df_silver.count())

valid_dates = df_silver.filter(
    col("date_added").isNotNull()
).count()

print("Valid Silver dates:", valid_dates)

if df_silver.count() == 6232 and valid_dates > 0:
    (
        df_silver.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
    )
    print("Silver table updated successfully!")
else:
    print("Validation failed. Silver table not overwritten.")


Silver records: 6232
Valid Silver dates: 6221
Silver table updated successfully!
